<div dir="rtl">

# NB00-M01 — Environment and Baseline

## هدف
این Notebook نقطه شروع رسمی فاز یک حرفه‌ای QRAG روی شاخه qrag-phase1 است.

در این مرحله هیچ Ingestion جدید، Embedding جدید، Retrieval جدید یا تغییر معنایی در داده انجام نمی‌شود. هدف فقط ثبت Baseline قابل استناد از محیط اجرا، Repository، داده‌های موجود و میراث نسخه آزمایشی قبلی است.

### ورودی‌ها
- Repository فعلی quran-rag
- شاخه qrag-phase1
- فایل‌های موجود در data
- Notebookهای قدیمی 01 تا 09
- فایل‌های تنظیمات موجود

### خروجی‌ها
- reports/phase1/environment_baseline_v1.json
- reports/phase1/environment_baseline_acceptance_v1.json

### خارج از Scope
دانلود منبع جدید، تغییر متن قرآن، Embedding، Vector DB، تماس API پولی و پذیرش تفسیر یا حدیث.

### معیار PASS
Branch، Commit، Root، ساختار Repository و فایل‌های ورودی قابل شناسایی باشند و Report پس از Reload مستقل Validate شود.

تا قبل از Restart Kernel → Run All روی clone واقعی Repository وضعیت رسمی PENDING_LOCAL_RUN_ALL است.

</div>

<div dir="rtl">

# NB00-M02 — برنامه اجرا

1. ثبت Environment بدون نمایش Secret.
2. کشف Dynamic Project Root.
3. ثبت وضعیت Git.
4. موجودی مسیرها و فایل‌های کلیدی.
5. Audit سلامت Notebookهای Legacy.
6. ثبت وضعیت داده‌های قرآن و Embedding موجود.
7. بررسی Dependency و Environment contract.
8. ساخت Baseline Report.
9. Reload و Validation مستقل.
10. ساخت Acceptance Report.

</div>

In [ ]:
# NB00-C01 — Imports and runtime environment
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

environment_info = {
    "python_version": sys.version.split()[0],
    "python_executable": sys.executable,
    "operating_system": platform.system(),
    "os_release": platform.release(),
    "machine": platform.machine(),
    "platform": platform.platform(),
    "execution_time_utc": datetime.now(timezone.utc).isoformat(),
}

for key, value in environment_info.items():
    print(f"{key}: {value}")

<div dir="rtl">

# NB00-M03 — کشف Root و Helperهای فاز یک

Root با مسیر شخصی Hard-code نمی‌شود. Notebook از مسیر فعلی به Parentها حرکت می‌کند و وجود .git، notebooks و data را بررسی می‌کند.

</div>

In [ ]:
# NB00-C02 — Project root discovery and helper import
current_directory = Path.cwd().resolve()
project_root = None

for candidate in [current_directory, *current_directory.parents]:
    if (
        (candidate / ".git").exists()
        and (candidate / "notebooks").exists()
        and (candidate / "data").exists()
    ):
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "QRAG project root could not be identified. "
        "Run this notebook from inside a real clone of the repository."
    )

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from qrag_phase1.foundation import (
    collect_git_info,
    collect_repository_inventory,
    sha256_file,
    write_json_atomic,
)

print("Project root:", project_root)

<div dir="rtl">

# NB00-M04 — Git Baseline

Branch، Commit و Working Tree ثبت می‌شوند. Dirty بودن به‌تنهایی Fail نیست ولی باید در Report دیده شود.

</div>

In [ ]:
# NB00-C03 — Git baseline
git_info = collect_git_info(project_root)

assert git_info["branch"], "Git branch could not be identified."
assert git_info["commit_sha"], "Git commit could not be identified."

print(json.dumps(git_info, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB00-M05 — Repository Inventory

مسیرهای پایه و فایل‌های کلیدی بررسی می‌شوند. در این مرحله هیچ فایل موجودی حذف یا بازنویسی نمی‌شود.

</div>

In [ ]:
# NB00-C04 — Repository inventory
required_paths = [
    ".git",
    "notebooks",
    "data",
    "docs/phase1",
]

tracked_key_files = [
    "app.py",
    "requirements.txt",
    ".env.example",
    "data/raw/quran_arabic_uthmani.txt",
    "data/raw/quran_fa_fooladvand.txt",
    "data/raw/quran_fa_ansarian.txt",
    "data/embeddings/quran_embeddings.npy",
]

repository_inventory = collect_repository_inventory(
    project_root=project_root,
    required_paths=required_paths,
    key_files=tracked_key_files,
)

assert repository_inventory["required_paths_ok"], (
    "Repository base structure is incomplete."
)

print(json.dumps(repository_inventory, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB00-M06 — Legacy Notebook Health

Notebookهای 01 تا 09 فقط Baseline قبلی هستند. تعداد Cell، Error Output و Code Cell اجرا‌نشده ثبت می‌شود.

</div>

In [ ]:
# NB00-C05 — Legacy notebook health audit
legacy_notebooks = [
    "01_download_quran_data.ipynb",
    "02_validate_quran_data.ipynb",
    "03_clean_quran_data.ipynb",
    "04_build_embeddings.ipynb",
    "05_test_retrieval.ipynb",
    "06_build_rag_pipeline.ipynb",
    "07_package_rag_pipeline.ipynb",
    "08_evaluate_rag_value.ipynb",
    "09_build_streamlit_quran_rag_app.ipynb",
]

legacy_health = []

for notebook_name in legacy_notebooks:
    notebook_path = project_root / "notebooks" / notebook_name

    if not notebook_path.exists():
        legacy_health.append({
            "notebook": notebook_name,
            "exists": False,
        })
        continue

    with notebook_path.open("r", encoding="utf-8") as handle:
        notebook_data = json.load(handle)

    cells = notebook_data.get("cells", [])
    code_cells = [cell for cell in cells if cell.get("cell_type") == "code"]
    markdown_cells = [
        cell for cell in cells if cell.get("cell_type") == "markdown"
    ]

    error_outputs = 0
    unexecuted_code_cells = 0

    for cell in code_cells:
        if cell.get("execution_count") is None:
            unexecuted_code_cells += 1

        for output in cell.get("outputs", []):
            if output.get("output_type") == "error":
                error_outputs += 1

    legacy_health.append({
        "notebook": notebook_name,
        "exists": True,
        "cell_count": len(cells),
        "code_cell_count": len(code_cells),
        "markdown_cell_count": len(markdown_cells),
        "unexecuted_code_cells": unexecuted_code_cells,
        "error_outputs": error_outputs,
    })

print(json.dumps(legacy_health, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB00-M07 — Baseline Data Inventory

فایل‌های موجود فقط Inventory می‌شوند. وجود فایل به معنی پذیرش Canonical یا حقوقی آن نیست.

</div>

In [ ]:
# NB00-C06 — Baseline data inventory
baseline_data_files = {
    "quran_arabic_uthmani": "data/raw/quran_arabic_uthmani.txt",
    "translation_fooladvand": "data/raw/quran_fa_fooladvand.txt",
    "translation_ansarian": "data/raw/quran_fa_ansarian.txt",
    "legacy_embeddings": "data/embeddings/quran_embeddings.npy",
    "legacy_processed_dataset": "data/processed/quran_dataset_clean.json",
}

data_inventory = {}

for key, relative_path in baseline_data_files.items():
    file_path = project_root / relative_path
    entry = {
        "path": relative_path,
        "exists": file_path.is_file(),
    }

    if file_path.is_file():
        entry["size_bytes"] = file_path.stat().st_size
        entry["sha256"] = sha256_file(file_path)

    data_inventory[key] = entry

print(json.dumps(data_inventory, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB00-M08 — Dependency و Secret Contract

مقدار Secret چاپ نمی‌شود. فقط وجود متغیرها و Dependencyهای Declare‌شده ثبت می‌شود.

</div>

In [ ]:
# NB00-C07 — Dependency and environment contract
requirements_path = project_root / "requirements.txt"
declared_requirements = []

if requirements_path.is_file():
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#"):
            declared_requirements.append(line)

environment_contract = {
    "ARVAN_AI_API_KEY": bool(os.getenv("ARVAN_AI_API_KEY", "").strip()),
    "ARVAN_CHAT_URL": bool(os.getenv("ARVAN_CHAT_URL", "").strip()),
    "ARVAN_CHAT_MODEL": bool(os.getenv("ARVAN_CHAT_MODEL", "").strip()),
}

dependency_contract = {
    "requirements_file_exists": requirements_path.is_file(),
    "declared_requirement_count": len(declared_requirements),
    "declared_requirements": declared_requirements,
}

print(json.dumps({
    "dependency_contract": dependency_contract,
    "environment_variable_presence": environment_contract,
}, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB00-M09 — ساخت Baseline Report

Report فقط وضعیت اندازه‌گیری‌شده را ذخیره می‌کند.

</div>

In [ ]:
# NB00-C08 — Build and persist baseline report
reports_directory = project_root / "reports" / "phase1"
reports_directory.mkdir(parents=True, exist_ok=True)

baseline_report_path = (
    reports_directory / "environment_baseline_v1.json"
)

baseline_report = {
    "schema_version": "qrag_environment_baseline_v1",
    "notebook": "notebooks/phase1/00_environment_and_baseline.ipynb",
    "phase": "qrag-phase1",
    "environment": environment_info,
    "git": git_info,
    "repository_inventory": repository_inventory,
    "legacy_notebook_health": legacy_health,
    "data_inventory": data_inventory,
    "dependency_contract": dependency_contract,
    "environment_variable_presence": environment_contract,
    "interpretation": {
        "legacy_assets_are_accepted_phase1_sources": False,
        "legacy_embeddings_are_frozen_reference_only": True,
        "source_acceptance_requires_downstream_notebooks": True,
    },
}

write_json_atomic(
    baseline_report_path,
    baseline_report,
)

print("Baseline report:", baseline_report_path.relative_to(project_root))

<div dir="rtl">

# NB00-M10 — Reload Validation

Report از Disk دوباره خوانده می‌شود.

</div>

In [ ]:
# NB00-C09 — Independent reload validation
with baseline_report_path.open("r", encoding="utf-8") as handle:
    reloaded_baseline = json.load(handle)

assert reloaded_baseline["schema_version"] == "qrag_environment_baseline_v1"
assert reloaded_baseline["phase"] == "qrag-phase1"
assert reloaded_baseline["git"]["branch"] == "qrag-phase1"
assert reloaded_baseline["repository_inventory"]["required_paths_ok"] is True
assert len(reloaded_baseline["legacy_notebook_health"]) == 9

for key in (
    "quran_arabic_uthmani",
    "translation_fooladvand",
    "translation_ansarian",
    "legacy_embeddings",
):
    assert reloaded_baseline["data_inventory"][key]["exists"] is True

print("NB00 reload validation: PASS")

<div dir="rtl">

# NB00-M11 — Final Acceptance

NB00 اجازه ورود به Source Policy را می‌دهد ولی Legacy assetها را Source پذیرفته‌شده اعلام نمی‌کند.

</div>

In [ ]:
# NB00-C10 — Final acceptance report
failed_checks = []
warnings = []

if git_info["branch"] != "qrag-phase1":
    failed_checks.append("WRONG_BRANCH")

if not repository_inventory["required_paths_ok"]:
    failed_checks.append("REPOSITORY_STRUCTURE_INCOMPLETE")

if dependency_contract["declared_requirement_count"] == 0:
    warnings.append("REQUIREMENTS_NOT_DECLARED")

legacy_unexecuted = sum(
    item.get("unexecuted_code_cells", 0)
    for item in legacy_health
)
legacy_errors = sum(
    item.get("error_outputs", 0)
    for item in legacy_health
)

if legacy_unexecuted:
    warnings.append(
        f"LEGACY_UNEXECUTED_CODE_CELLS:{legacy_unexecuted}"
    )

if legacy_errors:
    warnings.append(
        f"LEGACY_ERROR_OUTPUTS:{legacy_errors}"
    )

acceptance_status = (
    "FAIL"
    if failed_checks
    else (
        "PASS_WITH_LIMITATIONS"
        if warnings
        else "PASS"
    )
)

acceptance_report = {
    "schema_version": "qrag_nb00_acceptance_v1",
    "objective_status": (
        "ACHIEVED" if not failed_checks else "NOT_ACHIEVED"
    ),
    "acceptance_status": acceptance_status,
    "failed_checks": failed_checks,
    "warnings": warnings,
    "baseline_report": str(
        baseline_report_path.relative_to(project_root)
    ),
    "baseline_report_sha256": sha256_file(
        baseline_report_path
    ),
    "next_notebook": (
        "notebooks/phase1/01_source_policy_rights_and_authority.ipynb"
        if not failed_checks
        else None
    ),
}

acceptance_report_path = (
    reports_directory / "environment_baseline_acceptance_v1.json"
)

write_json_atomic(
    acceptance_report_path,
    acceptance_report,
)

with acceptance_report_path.open("r", encoding="utf-8") as handle:
    acceptance_reload = json.load(handle)

assert acceptance_reload == acceptance_report

print(json.dumps(acceptance_report, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB00-M12 — جمع‌بندی

این Notebook فقط Baseline را ثبت می‌کند. پس از اجرای واقعی باید Branch، Commit، داده‌های موجود، وضعیت Notebookهای Legacy، Dependencyها و محدودیت‌های نسخه قبلی روشن باشند.

</div>